# ShopFlow — Phase 4: Performance + Security
### Result cache · RBAC · dynamic data masking · row access policies

---

### What this notebook does

Two domains in one phase — both critical for production Snowflake:

**Performance:**

| Topic | What you'll do |
|-------|----------------|
| Result cache | Demonstrate zero-credit repeated queries — run the same query twice, compare execution times |
| Query profiling | Read `QUERY_HISTORY_BY_SESSION` — understand what Snowflake reports about every query |

**Security:**

| Topic | What you'll do |
|-------|----------------|
| RBAC | Create 3 project roles with different privilege levels |
| Dynamic data masking | Mask a sensitive column — different roles see different values |
| Row access policy | Restrict row visibility — `SHOPFLOW_SELLER` sees only their own orders |

### Why security belongs in the data warehouse

Application-level security can be bypassed — a developer removes a WHERE clause, or an analyst queries via Python directly. Snowflake policies are enforced **below** the application layer. `SELECT *` returns only what the current role is allowed to see, regardless of which tool or language is used.

### AWS equivalents

| Snowflake | AWS |
|-----------|-----|
| RBAC roles + GRANT | IAM roles + policies |
| Dynamic data masking | No direct equivalent — typically done at app layer |
| Row access policy | RLS in Redshift (similar concept) |
| Result cache | ElastiCache / DAX — but Snowflake does it automatically |

---

### Run cells one at a time — top to bottom

Each cell = one statement = one visible result.

---

### Key Snowflake concepts in this notebook

| Topic | Cells |
|-------|-------|
| Result cache — 3 cache types, 24-hour TTL | Cell 2–3 |
| `QUERY_HISTORY_BY_SESSION` | Cell 3 |
| Role hierarchy — system roles | Cell 4 |
| `CREATE ROLE` + `GRANT PRIVILEGE ON OBJECT TO ROLE` | Cell 5–7 |
| `GRANT ROLE TO USER` vs `GRANT ROLE TO ROLE` | Cell 7 |
| Dynamic data masking — `CREATE MASKING POLICY` | Cell 8–10 |
| Row access policy — `CREATE ROW ACCESS POLICY` | Cell 11–13 |
| `ALTER TABLE ... ADD ROW ACCESS POLICY` | Cell 13 |
| Testing policies — `USE ROLE` to switch context | Cell 9, 12 |


---
## Cell 1 — Set context

Phase 4 works across multiple schemas and roles. We start as `SYSADMIN` for object creation, then switch roles deliberately to test policies. Always confirm context before each section.


In [ ]:
USE ROLE      SYSADMIN;
--USE WAREHOUSE SHOPFLOW_WH;
USE DATABASE  SHOPFLOW_DB;
USE SCHEMA    SHOPFLOW_ANALYTICS;


In [ ]:
SELECT
    CURRENT_ROLE()      AS role,
    CURRENT_WAREHOUSE() AS warehouse,
    CURRENT_DATABASE()  AS database,
    CURRENT_SCHEMA()    AS schema;


---
## Cells 2–3 — Result cache demonstration

### Snowflake's three cache layers

Snowflake has three distinct caching mechanisms, each at a different level:

| Cache | What it stores | Who shares it | TTL | Credits |
|-------|---------------|---------------|-----|---------|
| **Metadata cache** | Object metadata — row counts, min/max values, `SHOW` results | All users | Until stale | 0 |
| **Result cache** | Complete query results | All users on the account | 24 hours | 0 |
| **Local disk cache** | Recently scanned micro-partition data | Per warehouse | Until warehouse suspends | 0 (already paid) |

The result cache is the most powerful for cost control. If any user on the account ran the **exact same query** against the **same data** within the last 24 hours — Snowflake returns the cached result instantly. Zero credits. Zero warehouse spin-up.

### Conditions that invalidate the result cache

The cache is invalidated when:
- The underlying table data changes (any DML — INSERT, UPDATE, DELETE, MERGE)
- The table is re-clustered
- More than 24 hours have passed
- The query is not byte-for-byte identical (even a comment difference breaks it)
- Session parameters that affect results change (e.g. `TIMESTAMP_OUTPUT_FORMAT`)

> **🎯 SnowPro Tip: Result Cache Rules**
>
> Three things to know for the exam:
> 1. Result cache is **account-level** — shared across all users and all warehouses. User A's query populates the cache for User B.
> 2. The warehouse does **not** need to be running to return a cached result. This is one of the most tested facts.
> 3. `ALTER SESSION SET USE_CACHED_RESULT = FALSE` disables result cache for the current session — useful for benchmarking.


---
## Cell 2 — Run the GMV query — first execution

Run this cell. Note the execution time from the result header in Snowsight.

Then run Cell 3 immediately after — it is the **exact same query**. Compare the times.


In [ ]:
-- First execution — hits the warehouse, scans data, uses credits
-- Note the execution time shown in the Snowsight result header
SELECT
    DATE_TRUNC('month', order_purchase_ts)  AS order_month,
    COUNT(DISTINCT order_id)                AS orders,
    ROUND(SUM(total_item_value), 2)         AS gmv_brl
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS
WHERE order_status = 'delivered'
GROUP BY 1
ORDER BY 1;


---
## Cell 3 — Run the same query again — result cache hit

Identical query. Snowflake returns from cache — 0 credits, near-instant response.

After running, query `QUERY_HISTORY_BY_SESSION` to see both executions side by side.


In [ ]:
-- Second execution — identical query returns from result cache
-- 0 credits · no warehouse needed · typically < 100ms
SELECT
    DATE_TRUNC('month', order_purchase_ts)  AS order_month,
    COUNT(DISTINCT order_id)                AS orders,
    ROUND(SUM(total_item_value), 2)         AS gmv_brl
FROM SHOPFLOW_ANALYTICS.FACT_ORDERS
WHERE order_status = 'delivered'
GROUP BY 1
ORDER BY 1;


In [ ]:
-- Compare both executions in QUERY_HISTORY
-- Look for 'result_cache_hit' column — TRUE on the second run
-- Also compare total_elapsed_time: first run vs second run
SELECT
    LEFT(query_text, 60)                        AS query_preview,
    TO_CHAR(start_time, 'HH24:MI:SS')           AS run_at,
    ROUND(total_elapsed_time / 1000.0, 3)       AS elapsed_seconds,
    execution_status,
    CASE WHEN query_tag = '' THEN 'n/a' ELSE query_tag END AS tag
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION(
    RESULT_LIMIT => 10
))
WHERE query_text ILIKE '%gmv_brl%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 4;


---
## Cells 4–7 — Role-Based Access Control (RBAC)

### Snowflake's built-in role hierarchy

Snowflake ships with 5 system roles arranged in a strict inheritance hierarchy. Higher roles inherit all privileges of lower ones:

```text
ACCOUNTADMIN          ← full account control — never use day-to-day
    └── SYSADMIN      ← creates and manages all objects
    └── SECURITYADMIN ← manages users, roles, grants
            └── USERADMIN  ← creates users and roles only
                    └── PUBLIC ← default for every user, minimal privileges
```

### The three ShopFlow project roles

We create three custom roles on top of this hierarchy — one per persona:

| Role | Who | What they can do |
|------|-----|-----------------|
| `SHOPFLOW_ENGINEER` | Data engineer (you) | Full access to all 3 schemas — read + write |
| `SHOPFLOW_ANALYST` | Business analyst | `SELECT` on `SHOPFLOW_ANALYTICS` only — no RAW, no STAGED |
| `SHOPFLOW_SELLER` | Seller partner | `SELECT` on `SHOPFLOW_ANALYTICS` — but only rows for their own `seller_id` |

### Two operations always required — both needed

```sql
-- 1. Give a role permission on an object
GRANT SELECT ON TABLE my_table TO ROLE my_role;

-- 2. Assign the role to a user so they can USE ROLE
GRANT ROLE my_role TO USER my_user;
```

Both steps are always required. A role with no users is useless. A user with no role grants can't USE ROLE.

> **🎯 SnowPro Tip: Object Ownership**
>
> The role that **creates** an object **owns** it by default. Ownership = full control including the ability to grant privileges to others. In production, objects should be owned by a service role (`SYSADMIN` or a dedicated object-owner role), not by a personal user account. If the person leaves, the object ownership doesn't follow them.


---
## Cell 4 — Explore existing roles

Before creating new roles, see what already exists in your account.


In [ ]:
-- See all roles in the account
-- Note the 5 system roles: ACCOUNTADMIN, SYSADMIN, SECURITYADMIN, USERADMIN, PUBLIC
SHOW ROLES;


In [ ]:
-- See what privileges your current role has on SHOPFLOW_DB
SHOW GRANTS ON DATABASE SHOPFLOW_DB;


---
## Cell 5 — Create the three project roles

We use `SECURITYADMIN` to create roles — this is its specific responsibility in the system role hierarchy. `SYSADMIN` creates objects. `SECURITYADMIN` manages access to those objects.


In [ ]:
USE ROLE SECURITYADMIN;

-- Role 1: Full data engineer access
CREATE ROLE IF NOT EXISTS SHOPFLOW_ENGINEER
    COMMENT = 'ShopFlow data engineer — full access to all 3 schemas';

-- Role 2: Read-only analyst on ANALYTICS schema only
CREATE ROLE IF NOT EXISTS SHOPFLOW_ANALYST
    COMMENT = 'ShopFlow analyst — SELECT on SHOPFLOW_ANALYTICS only';

-- Role 3: Seller — ANALYTICS read-only + row-level filter applied
CREATE ROLE IF NOT EXISTS SHOPFLOW_SELLER
    COMMENT = 'ShopFlow seller — SELECT on ANALYTICS, own rows only via row access policy';

SHOW ROLES LIKE 'SHOPFLOW%';


---
## Cell 6 — Grant warehouse and database privileges

A role needs explicit `USAGE` grants on warehouse, database, and schema before it can query any table. These grants are hierarchical — missing any one of them blocks access even if the table grant exists.

```text
USAGE on warehouse    ← without this, queries cannot execute
USAGE on database     ← without this, objects aren't visible
USAGE on schema       ← without this, tables aren't accessible
SELECT on table       ← the actual data permission
```


In [ ]:
USE ROLE SECURITYADMIN;

-- ── SHOPFLOW_ENGINEER ──────────────────────────────────────────────────────
-- Warehouse usage
GRANT USAGE ON WAREHOUSE SHOPFLOW_WH TO ROLE SHOPFLOW_ENGINEER;

-- Database + all 3 schemas
GRANT USAGE ON DATABASE SHOPFLOW_DB TO ROLE SHOPFLOW_ENGINEER;
GRANT USAGE ON SCHEMA SHOPFLOW_DB.SHOPFLOW_RAW       TO ROLE SHOPFLOW_ENGINEER;
GRANT USAGE ON SCHEMA SHOPFLOW_DB.SHOPFLOW_STAGED    TO ROLE SHOPFLOW_ENGINEER;
GRANT USAGE ON SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS TO ROLE SHOPFLOW_ENGINEER;

-- Full SELECT on all current + future tables in all schemas
GRANT SELECT ON ALL TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_RAW       TO ROLE SHOPFLOW_ENGINEER;
GRANT SELECT ON ALL TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_STAGED     TO ROLE SHOPFLOW_ENGINEER;
GRANT SELECT ON ALL TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS  TO ROLE SHOPFLOW_ENGINEER;

GRANT SELECT ON FUTURE TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_RAW       TO ROLE SHOPFLOW_ENGINEER;
GRANT SELECT ON FUTURE TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_STAGED     TO ROLE SHOPFLOW_ENGINEER;
GRANT SELECT ON FUTURE TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS  TO ROLE SHOPFLOW_ENGINEER;

-- ── SHOPFLOW_ANALYST ───────────────────────────────────────────────────────
-- ANALYTICS schema only — no access to RAW or STAGED
GRANT USAGE ON WAREHOUSE SHOPFLOW_WH              TO ROLE SHOPFLOW_ANALYST;
GRANT USAGE ON DATABASE SHOPFLOW_DB               TO ROLE SHOPFLOW_ANALYST;
GRANT USAGE ON SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS TO ROLE SHOPFLOW_ANALYST;
GRANT SELECT ON ALL TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS TO ROLE SHOPFLOW_ANALYST;
GRANT SELECT ON FUTURE TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS TO ROLE SHOPFLOW_ANALYST;

-- ── SHOPFLOW_SELLER ────────────────────────────────────────────────────────
-- Same grants as ANALYST — row access policy added in Cell 11 restricts rows
GRANT USAGE ON WAREHOUSE SHOPFLOW_WH              TO ROLE SHOPFLOW_SELLER;
GRANT USAGE ON DATABASE SHOPFLOW_DB               TO ROLE SHOPFLOW_SELLER;
GRANT USAGE ON SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS TO ROLE SHOPFLOW_SELLER;
GRANT SELECT ON ALL TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS TO ROLE SHOPFLOW_SELLER;
GRANT SELECT ON FUTURE TABLES IN SCHEMA SHOPFLOW_DB.SHOPFLOW_ANALYTICS TO ROLE SHOPFLOW_SELLER;

SELECT 'Privileges granted to all 3 roles' AS status;


---
## Cell 7 — Assign roles to your user

`GRANT ROLE TO USER` is a separate operation from `GRANT PRIVILEGE TO ROLE`. Both are always required. Privileges go to roles. Roles go to users.

Replace `YOUR_USERNAME` with your actual Snowflake username (visible in the top-right of Snowsight, or from `SELECT CURRENT_USER()`).


In [ ]:
USE ROLE SECURITYADMIN;

-- Replace YOUR_USERNAME with your actual Snowflake username
-- e.g. GRANT ROLE SHOPFLOW_ENGINEER TO USER PLATYPUS07;
GRANT ROLE SHOPFLOW_ENGINEER TO USER YOUR_USERNAME;
GRANT ROLE SHOPFLOW_ANALYST  TO USER YOUR_USERNAME;
GRANT ROLE SHOPFLOW_SELLER   TO USER YOUR_USERNAME;

-- Confirm all 3 roles are now assigned to your user
SHOW GRANTS TO USER YOUR_USERNAME;


In [ ]:
-- Verify: confirm grants on the warehouse for the project roles
SHOW GRANTS ON WAREHOUSE SHOPFLOW_WH;


---
## Cell 8 — Test role isolation

Switch to `SHOPFLOW_ANALYST` and try to query `SHOPFLOW_RAW`. It should fail — analysts have no access to the RAW schema. This confirms the isolation is working.

Switch back to `SYSADMIN` after — you need it for the rest of this notebook.


In [ ]:
-- Switch to ANALYST role
USE ROLE SHOPFLOW_ANALYST;
USE WAREHOUSE SHOPFLOW_WH;

-- This should SUCCEED — ANALYST has SELECT on ANALYTICS
SELECT COUNT(*) AS fact_rows FROM SHOPFLOW_DB.SHOPFLOW_ANALYTICS.FACT_ORDERS;


In [ ]:
-- This should FAIL — ANALYST has no USAGE on SHOPFLOW_RAW
-- Expected error: "Object 'SHOPFLOW_DB.SHOPFLOW_RAW.RAW_ORDERS' does not exist"
SELECT COUNT(*) AS raw_rows FROM SHOPFLOW_DB.SHOPFLOW_RAW.RAW_ORDERS;


In [ ]:
-- Switch back to SYSADMIN for the rest of the notebook
USE ROLE SYSADMIN;
USE DATABASE SHOPFLOW_DB;
USE SCHEMA SHOPFLOW_ANALYTICS;
SELECT CURRENT_ROLE() AS back_to;


---
## Cells 9–10 — Dynamic data masking

### What is dynamic data masking?

A **masking policy** is a SQL function applied to a column at query time. Different roles see different values for the same column — without any application code changes, without duplicating data, and without the user knowing masking is applied.

```text
SHOPFLOW_ENGINEER queries customer_state:   → 'SP'           (real value)
SHOPFLOW_ANALYST  queries customer_state:   → '**'           (masked)
SHOPFLOW_SELLER   queries customer_state:   → '**'           (masked)
```

The masking happens inside Snowflake's query engine. `SELECT *` returns masked values. There is no way for the analyst to bypass it.

### What we'll mask

`DIM_CUSTOMERS` has `city` and `state` — not PII in the strict sense, but useful for demonstrating masking. In a real system you'd mask email, phone number, or any column containing personal data.

### Masking policy syntax

```sql
CREATE OR REPLACE MASKING POLICY policy_name
    AS (val DATATYPE) RETURNS DATATYPE ->
    CASE
        WHEN CURRENT_ROLE() = 'privileged_role' THEN val   -- show real value
        ELSE '**'                                           -- return masked value
    END;
```

The policy is a SQL expression. The `val` parameter receives the column value. Return the real value for privileged roles, return a safe substitute for everyone else.

> **🎯 SnowPro Tip: Masking Policy Rules**
>
> Key facts for the exam:
> - One masking policy per column — you cannot stack multiple policies on the same column
> - The policy function is evaluated **per row, per query** — small overhead but transparent
> - `INFORMATION_SCHEMA.POLICY_REFERENCES` shows which columns have policies applied
> - Only `POLICY_ADMIN` or `ACCOUNTADMIN` can create masking policies by default — in this notebook we use `SYSADMIN` which has sufficient privileges on the trial account


---
## Cell 9 — Create and apply a masking policy


In [ ]:
USE ROLE SYSADMIN;
USE SCHEMA SHOPFLOW_ANALYTICS;

-- Create masking policy for VARCHAR columns (city, state)
-- SHOPFLOW_ENGINEER sees real values
-- All other roles see '**'
CREATE OR REPLACE MASKING POLICY shopflow_varchar_mask
    AS (val VARCHAR) RETURNS VARCHAR ->
    CASE
        WHEN CURRENT_ROLE() = 'SHOPFLOW_ENGINEER' THEN val
        ELSE '**'
    END
    COMMENT = 'Masks VARCHAR PII columns from non-engineer roles';

-- Confirm policy was created
SHOW MASKING POLICIES IN SCHEMA SHOPFLOW_ANALYTICS;


In [ ]:
-- Apply the masking policy to DIM_CUSTOMERS.city and DIM_CUSTOMERS.state
ALTER TABLE SHOPFLOW_ANALYTICS.DIM_CUSTOMERS
    MODIFY COLUMN city  SET MASKING POLICY shopflow_varchar_mask;

ALTER TABLE SHOPFLOW_ANALYTICS.DIM_CUSTOMERS
    MODIFY COLUMN state SET MASKING POLICY shopflow_varchar_mask;

SELECT 'Masking policy applied to DIM_CUSTOMERS.city and .state' AS status;


---
## Cell 10 — Test the masking policy

Run the same query under two different roles. The data is identical — the masking policy changes what each role sees.


In [ ]:
-- As SHOPFLOW_ENGINEER — should see real city and state values
USE ROLE SHOPFLOW_ENGINEER;
USE WAREHOUSE SHOPFLOW_WH;

SELECT customer_id, city, state
FROM SHOPFLOW_DB.SHOPFLOW_ANALYTICS.DIM_CUSTOMERS
LIMIT 5;


In [ ]:
-- As SHOPFLOW_ANALYST — city and state should show '**'
USE ROLE SHOPFLOW_ANALYST;

SELECT customer_id, city, state
FROM SHOPFLOW_DB.SHOPFLOW_ANALYTICS.DIM_CUSTOMERS
LIMIT 5;


In [ ]:
-- Switch back to SYSADMIN
USE ROLE SYSADMIN;
USE DATABASE SHOPFLOW_DB;
USE SCHEMA SHOPFLOW_ANALYTICS;
SELECT CURRENT_ROLE() AS back_to;


---
## Cells 11–13 — Row access policy

### What is a row access policy?

A **row access policy** is a boolean SQL function applied to a table that controls which rows are visible to the current session. Every query against the table has an invisible `WHERE` clause injected by Snowflake:

```sql
-- What the analyst writes:
SELECT * FROM FACT_ORDERS;

-- What Snowflake actually executes (conceptually):
SELECT * FROM FACT_ORDERS
WHERE <row_access_policy_function>(seller_id) = TRUE;
```

The policy function receives column values and returns `TRUE` (row is visible) or `FALSE` (row is hidden).

### The ShopFlow seller use case

`SHOPFLOW_SELLER` represents a seller partner who logs in to query their own sales data. They should see:
- ✅ Rows where `seller_id` matches their Snowflake username
- ❌ All other sellers' rows

`SHOPFLOW_ENGINEER` and `SHOPFLOW_ANALYST` should see all rows.

### Masking policy vs row access policy

| | Masking policy | Row access policy |
|--|---------------|-------------------|
| What it controls | Column values | Which rows appear |
| Applied to | A column | A table |
| Returns | The column value (possibly modified) | Boolean (row visible or not) |
| Use case | Hide PII values | Multi-tenant row isolation |

> **🎯 SnowPro Tip: Row Access Policy**
>
> - One row access policy per table — cannot stack multiple policies
> - The policy is applied **before** any user-specified WHERE clause — the user cannot bypass it
> - `CURRENT_USER()` inside the policy returns the username of whoever is running the query — this is how per-seller filtering works without managing a separate mapping table
> - Row access policies can also reference a mapping table via a subquery for more complex scenarios (e.g. a user-to-tenant mapping table)


---
## Cell 11 — Create the row access policy


In [ ]:
USE ROLE SYSADMIN;
USE SCHEMA SHOPFLOW_ANALYTICS;

-- Row access policy for FACT_ORDERS on seller_id column
-- Logic:
--   SHOPFLOW_ENGINEER or SHOPFLOW_ANALYST → see all rows (TRUE)
--   SHOPFLOW_SELLER → see only rows where seller_id = their username
--   Any other role → see nothing (FALSE)
CREATE OR REPLACE ROW ACCESS POLICY shopflow_seller_filter
    AS (seller_id VARCHAR) RETURNS BOOLEAN ->
    CASE
        WHEN CURRENT_ROLE() IN ('SHOPFLOW_ENGINEER', 'SHOPFLOW_ANALYST', 'SYSADMIN')
            THEN TRUE
        WHEN CURRENT_ROLE() = 'SHOPFLOW_SELLER'
            THEN seller_id = CURRENT_USER()
        ELSE FALSE
    END
    COMMENT = 'Sellers see only their own rows in FACT_ORDERS';

SHOW ROW ACCESS POLICIES IN SCHEMA SHOPFLOW_ANALYTICS;


In [ ]:
-- Apply the row access policy to FACT_ORDERS on the seller_id column
ALTER TABLE SHOPFLOW_ANALYTICS.FACT_ORDERS
    ADD ROW ACCESS POLICY shopflow_seller_filter ON (seller_id);

SELECT 'Row access policy applied to FACT_ORDERS.seller_id' AS status;


---
## Cell 12 — Test the row access policy

`SHOPFLOW_ANALYST` should see all rows. `SHOPFLOW_SELLER` should see 0 rows — because no `seller_id` in `FACT_ORDERS` matches the literal Snowflake username `'SHOPFLOW_SELLER'` (in a real deployment, the seller's Snowflake username would match their `seller_id`).

This demonstrates the policy is active and working correctly.


In [ ]:
-- SHOPFLOW_ANALYST — should see all ~112k rows
USE ROLE SHOPFLOW_ANALYST;
USE WAREHOUSE SHOPFLOW_WH;

SELECT COUNT(*) AS visible_rows,
       'SHOPFLOW_ANALYST — expects all rows' AS note
FROM SHOPFLOW_DB.SHOPFLOW_ANALYTICS.FACT_ORDERS;


In [ ]:
-- SHOPFLOW_SELLER — sees only rows where seller_id = CURRENT_USER()
-- = 'SHOPFLOW_SELLER' which doesn't match any real seller_id → 0 rows
-- In production: seller's Snowflake username = their seller_id in the data
USE ROLE SHOPFLOW_SELLER;

SELECT COUNT(*) AS visible_rows,
       CURRENT_USER() AS my_username,
       'SHOPFLOW_SELLER — expects 0 (username does not match any seller_id)' AS note
FROM SHOPFLOW_DB.SHOPFLOW_ANALYTICS.FACT_ORDERS;


In [ ]:
-- Switch back to SYSADMIN
USE ROLE SYSADMIN;
USE DATABASE SHOPFLOW_DB;
USE SCHEMA SHOPFLOW_ANALYTICS;
SELECT CURRENT_ROLE() AS back_to;


---
## Cell 13 — Verify all policies are in place

`INFORMATION_SCHEMA.POLICY_REFERENCES` shows every column and table that has a policy applied. Use this to audit what is protected before handing access to other users.


In [ ]:
-- List all masking policies applied in SHOPFLOW_ANALYTICS
SELECT
    policy_name,
    policy_kind,
    ref_entity_name     AS table_name,
    ref_column_name     AS column_name,
    policy_status
FROM TABLE(INFORMATION_SCHEMA.POLICY_REFERENCES(
    policy_name => 'SHOPFLOW_ANALYTICS.SHOPFLOW_VARCHAR_MASK'
));


In [ ]:
-- List all row access policies applied in SHOPFLOW_ANALYTICS
SELECT
    policy_name,
    policy_kind,
    ref_entity_name     AS table_name,
    ref_column_name     AS column_name,
    policy_status
FROM TABLE(INFORMATION_SCHEMA.POLICY_REFERENCES(
    policy_name => 'SHOPFLOW_ANALYTICS.SHOPFLOW_SELLER_FILTER'
));


---
## Cell 14 — Suspend warehouse

Always suspend explicitly at session end.


In [ ]:
USE ROLE SYSADMIN;
ALTER WAREHOUSE SHOPFLOW_WH SUSPEND;


In [ ]:
SHOW WAREHOUSES LIKE 'SHOPFLOW%';


---
## 🛠 Self-Guided Exploration:

Now that RBAC, masking, and row access policies are in place — test your understanding. Write these from scratch.

### Challenge 1 (Easy): Inspect your role privileges
**Objective:** Practice reading privilege metadata.

**Task:** Write a `SHOW` command that lists all privileges granted **to** the `SHOPFLOW_ANALYST` role. How many privilege rows do you see, and on which objects?

> 💡 **Think about it:** Is `USAGE ON WAREHOUSE` listed? Why does an analyst need that — what happens without it?


In [ ]:
-- Write your SQL for Challenge 1 here


### Challenge 2 (Medium): The masking bypass attempt
**Objective:** Confirm that masking cannot be bypassed by switching approaches.

**Task:** As `SHOPFLOW_ANALYST`, write a query that tries to see real `city` values using a subquery or CTE instead of a direct SELECT. Does the masking still apply?

> 💡 **Think about it:** The policy is enforced at the column level inside Snowflake's engine — not at the query syntax level. What does that mean for whether a subquery can bypass it?


In [ ]:
-- Write your SQL for Challenge 2 here
-- Hint: try something like:
-- WITH cte AS (SELECT customer_id, city FROM DIM_CUSTOMERS)
-- SELECT * FROM cte LIMIT 5;


### Challenge 3 (Hard): Role hierarchy check
**Objective:** Understand privilege inheritance in Snowflake.

**Task:** Write a query or SHOW command that answers: does `SYSADMIN` have access to `SHOPFLOW_ANALYTICS` tables even though we only granted privileges to `SHOPFLOW_ENGINEER`, `SHOPFLOW_ANALYST`, and `SHOPFLOW_SELLER`?

> 💡 **Think about it:** `SYSADMIN` created the tables and therefore **owns** them. What does object ownership mean for access — does the owner need an explicit GRANT?


In [ ]:
-- Write your SQL for Challenge 3 here


### Challenge 4 (Complex): Drop and recreate the masking policy
**Objective:** Practice the full policy lifecycle — drop, recreate with different logic, reapply.

**Task:** Drop `shopflow_varchar_mask` from `DIM_CUSTOMERS.city` only (leave `state` masked). Then recreate the policy with new logic: `SHOPFLOW_ANALYST` sees real city values, but all other non-engineer roles see `'REDACTED'` instead of `'**'`. Reapply to `city` only and verify both columns behave as expected under `SHOPFLOW_ANALYST`.

> ⚠️ **Watch out:** You must `UNSET` a policy from a column before you can drop the policy object itself — Snowflake will error if you try to drop a policy that is still in use.


In [ ]:
-- Write your SQL for Challenge 4 here


---
## Phase 4 complete

If Cell 3 shows `QUERY_HISTORY` with a near-zero second result on the second GMV run, Cell 8 confirms `SHOPFLOW_ANALYST` cannot query `SHOPFLOW_RAW`, Cell 10 shows `'**'` for city/state under analyst role, and Cell 12 shows 0 rows for `SHOPFLOW_SELLER` — your performance and security layer is fully operational.

### What you just built

```text
SHOPFLOW_DB
├── Roles
│   ├── SHOPFLOW_ENGINEER   ← full access, all 3 schemas
│   ├── SHOPFLOW_ANALYST    ← SELECT on ANALYTICS only
│   └── SHOPFLOW_SELLER     ← SELECT on ANALYTICS, own rows only
│
├── Masking policies
│   └── shopflow_varchar_mask
│       ├── → DIM_CUSTOMERS.city
│       └── → DIM_CUSTOMERS.state
│
└── Row access policies
    └── shopflow_seller_filter
        └── → FACT_ORDERS (on seller_id)
```

### Key Snowflake concepts you practised

| Topic | Covered |
|-------|---------|
| Result cache — 3 types, 24-hour TTL | ✅ |
| Result cache — account-level, no warehouse needed | ✅ |
| QUERY_HISTORY_BY_SESSION | ✅ |
| System role hierarchy | ✅ |
| CREATE ROLE + GRANT PRIVILEGE ON OBJECT TO ROLE | ✅ |
| GRANT ROLE TO USER | ✅ |
| GRANT ON FUTURE TABLES | ✅ |
| Dynamic data masking — CREATE MASKING POLICY | ✅ |
| ALTER TABLE ... MODIFY COLUMN SET MASKING POLICY | ✅ |
| Row access policy — CREATE ROW ACCESS POLICY | ✅ |
| ALTER TABLE ... ADD ROW ACCESS POLICY | ✅ |
| INFORMATION_SCHEMA.POLICY_REFERENCES | ✅ |
| CURRENT_USER() inside policy for per-user filtering | ✅ |

### Next — Phase 5: Streams, Tasks & Sharing

- `CREATE STREAM` on `FACT_ORDERS` — CDC captures every change
- `CREATE TASK` with CRON schedule — nightly aggregation, replaces EventBridge + Lambda
- `SYSTEM$STREAM_HAS_DATA()` — conditional task execution
- `CREATE SHARE` — partner queries live data, no file exports
- Snowflake Marketplace context — how public data sharing works
